`Запрос` — фильтруем по номеру месяца через `EXTRACT`, группируем по покупателю:

```sql
SELECT c.customer_id, c.customer_name, SUM(o.total_amount) AS total_amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE EXTRACT(MONTH FROM o.order_date) = 1
GROUP BY c.customer_id, c.customer_name
ORDER BY c.customer_id;
```
`ORDER BY customer_id с JOIN` гарантирует, что в отчёт попадут только покупатели, у которых есть хотя бы один заказ в январе (строки без заказов просто не создадутся).

Если хочется гибко подставлять месяц, заверните в функцию или передавайте параметр:

`WHERE EXTRACT(MONTH FROM o.order_date) = %s`   -- например 1 для января

По вашим данным (январь есть только в 2024-01-10 и 2024-01-22) результат будет:

| customer_id    | customer_name | total_amount     |
|:---------------|:-------------:|-----------------:|
| 1              |   John Smit   |    130           |
| 2              |   Emily White |    290           |



---

`Подробное объяснение`

Запись `ORDER BY c.customer_id;` сортирует строки результата по столбцу `customer_id` перед возвратом — и это единственное место в запросе, где порядок строк вообще определяется (`ORDER BY` выполняется в самую последнюю очередь, после фильтрации, группировки и отбора колонок).

`Как это работает`

Ключевая часть — `c.` — алиас (псевдоним) таблицы:

```sql
FROM orders o
JOIN customers c ON ...
```

- `o` — алиас для `orders`
- `c` — алиас для `customers`

Поэтому `c.customer_id` — это точно столбец `customer_id` из таблицы `customers` (а не из `orders`). Без алиаса можно было бы написать `ORDER BY customer_id`, но алиасы снимают неоднозначность, если в нескольких таблицах есть столбец с одним именем.

`Правила сортировки`

1. По возрастанию по умолчанию — `ASC`. Чтобы явно — `ORDER BY c.customer_id ASC`. В убывающую — `DESC`.

2. Cортировка по умолчанию зависит от локали для текста (`collation`), но для чисел — обычное числовое сравнение.

3. `NULL` идут в конец при `ASC`, в начало при `DESC` — если не указать `NULLS FIRST`/`NULLS LAST`.

4. `Стабильность`: если значения ключа совпадают, порядок строк между ними не гарантирован — нужно добавить второй ключ.


`Несколько ключей`

```sql
ORDER BY c.customer_id ASC, o.order_date DESC
```
Сначала сортировка по `customer_id`, а внутри одинаковых `customer_id` — по дате убывающе.


`В контексте запроса`

```sql
SELECT c.customer_id, c.customer_name, SUM(o.total_amount) AS total_amount
...
GROUP BY c.customer_id, c.customer_name
ORDER BY c.customer_id;
```
- `GROUP BY` разбил заказы по каждому покупателю, `SUM` посчитал сумму.

- `Итог` — покупатели в порядке возрастания id: 1, 10.

- `Важно`: столбец в `ORDER BY` либо должен быть в `GROUP BY`, либо быть в агрегатной функции (как`SUM`). Здесь     `customer_id` есть в `GROUP BY` — всё корректно. `customer_name` тоже в `GROUP BY`.

`Что было бы без ORDER BY`

- Порядок строк не гарантирован вообще — обычно совпадает с физическим порядком в индексе/таблице, но рассчитывать на это нельзя. Поэтому для предсказуемого вывода `ORDER BY` обязателен.

- `Нюанс`: если бы вы сортировали просто по `c.customer_id`, а нужно бы показать итоговую сумму по убыванию ("кто потратил больше") — тогда `ORDER BY total_amount DESC` или `ORDER BY SUM(o.total_amount) DESC`.